# 모듈 01: 첫 번째 그래프

모듈 00에서 LangGraph를 설치했습니다. 이제 실제로 그래프를 만들어봅니다.

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | LangGraph의 핵심 3요소(노드, 엣지, 상태) 이해 |
| 2 | 가장 단순한 그래프를 4단계로 조립 |
| 3 | LLM을 노드로 연결해 실제 AI 응답 받기 |
| 4 | 그래프 구조 시각화로 흐름 확인 |

예상 소요 시간: 약 30분  
전제 조건: 모듈 00 완료 (langgraph 설치, GOOGLE_API_KEY 설정)

## 학습 목표

1. `StateGraph`의 4단계 패턴(정의 → 노드 → 엣지 → 컴파일)을 설명할 수 있다
2. 노드 함수의 입출력 규칙을 이해한다: 상태(State)를 받아 변경사항만 딕셔너리로 반환
3. LLM을 노드 함수로 감싸서 그래프에 통합할 수 있다

## 전체 흐름

```
┌─────────────────────────────────────────────────┐
│                                                 │
│  [1] LangGraph 3대 요소 개념 이해               │
│       ↓                                         │
│  [2] Step 1: 상태(State) 정의                   │
│       ↓                                         │
│  [3] Step 2: 노드 함수 작성                     │
│       ↓                                         │
│  [4] Step 3: 그래프 조립 (add_node, add_edge)   │
│       ↓                                         │
│  [5] Step 4: 컴파일 + 실행 (invoke)             │
│       ↓                                         │
│  [6] LLM 노드로 업그레이드                      │
│                                                 │
└─────────────────────────────────────────────────┘
```

---

## 섹션 2: LangGraph 핵심 3요소

LangGraph는 세 가지 구성 요소로 이루어집니다.

| 요소 | 역할 | 비유 |
|------|------|------|
| **상태(State)** | 노드 간에 공유되는 데이터 | 열차의 화물칸 |
| **노드(Node)** | 실행할 함수 | 역 (역마다 화물을 처리) |
| **엣지(Edge)** | 노드 사이 연결 | 철로 (어느 역으로 갈지 결정) |

### 구조 다이어그램

```
START → [노드A] → [노드B] → END
         ↑           ↑
       (함수)      (함수)
         ↕           ↕
       State 공유 (딕셔너리)
```

### 노드 함수의 핵심 규칙

- **입력**: 현재 상태 전체를 받음 (`state: State`)
- **출력**: 변경된 부분만 딕셔너리로 반환 (`return {"key": value}`)

전체 상태를 반환하지 않아도 됩니다. 바꾸고 싶은 필드만 반환하면 LangGraph가 나머지를 유지합니다.

In [ ]:
# .env 로드 및 패키지 임포트
import os
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, END

# .env 경로: LangChain/.env 또는 LangGraph/.env
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
langgraph_root = os.path.dirname(notebook_dir)
project_root = os.path.dirname(langgraph_root)
langchain_env = os.path.join(project_root, 'LangChain', '.env')
langgraph_env = os.path.join(langgraph_root, '.env')

if os.path.exists(langchain_env):
    load_dotenv(langchain_env)
    print('[OK] LangChain/.env 로드 완료')
elif os.path.exists(langgraph_env):
    load_dotenv(langgraph_env)
    print('[OK] LangGraph/.env 로드 완료')
else:
    print('[FAIL] .env 파일을 찾을 수 없습니다')

api_key = os.getenv('GOOGLE_API_KEY')
print(f'[OK] GOOGLE_API_KEY: {"설정됨" if api_key else "[FAIL] 미설정"}')

---

## 섹션 3: 첫 번째 그래프 만들기

이제 4단계로 첫 번째 그래프를 조립합니다.

### 설계

```
START → [greet] → END
```

- 노드: `greet` (인사 메시지를 상태에 추가하는 함수)
- 상태: `{"message": str}` 형태의 딕셔너리

In [ ]:
# Step 1: 상태(State) 정의
# TypedDict로 상태의 '설계도'를 정의합니다
from typing import TypedDict

class State(TypedDict):
    message: str  # 노드들이 공유할 메시지 필드

# 상태는 그냥 딕셔너리입니다
initial_state: State = {"message": ""}
print(f'초기 상태: {initial_state}')
print(f'타입: {type(initial_state)}')

In [ ]:
# Step 2: 노드 함수 작성
# 노드 함수: State를 받아서 변경사항만 dict로 반환
def greet(state: State) -> dict:
    """인사 메시지를 상태에 추가하는 노드"""
    current = state.get("message", "")
    updated = current + "안녕하세요! LangGraph 첫 번째 그래프입니다."
    return {"message": updated}

# 함수 직접 테스트
test_state: State = {"message": ""}
result = greet(test_state)
print(f'노드 반환값: {result}')
print(f'타입: {type(result)}')
print()
print('노드 함수 규칙:')
print('  입력 → State 전체 (딕셔너리)')
print('  출력 → 변경사항만 딕셔너리로 반환')

### Step 3: 그래프 조립

`StateGraph`의 핵심 API 4개를 사용합니다.

| 메서드 | 역할 |
|--------|------|
| `add_node("이름", 함수)` | 그래프에 노드 등록 |
| `set_entry_point("이름")` | 실행 시작 노드 지정 |
| `add_edge("A", "B")` | A에서 B로 흐름 연결 |
| `add_edge("이름", END)` | 종료 엣지 추가 |

In [ ]:
# Step 3: 그래프 조립
# 그래프 생성 (상태 타입 지정)
graph = StateGraph(State)

# 노드 등록
graph.add_node("greet", greet)

# 진입점 설정
graph.set_entry_point("greet")

# 종료 엣지 추가
graph.add_edge("greet", END)

print('[OK] 그래프 정의 완료!')
print('구조: START → greet → END')

In [ ]:
# Step 4: 컴파일 및 실행
# 컴파일: 정의된 그래프를 실행 가능한 객체로 만듦
app = graph.compile()
print('[OK] 컴파일 완료!')

# 실행: 초기 상태를 넣으면 최종 상태가 반환됨
result = app.invoke({"message": ""})

print()
print('=== 실행 결과 ===')
print(f'최종 상태: {result}')
print(f'message: {result["message"]}')

---

## 섹션 4: LLM을 노드로 만들기

이제 단순 문자열 처리 대신, 실제 LLM을 노드로 만들어봅니다.

- 노드 함수 안에서 `ChatGoogleGenerativeAI`를 호출
- 상태의 `user_input` 필드를 읽어 LLM에 전달
- LLM 응답을 `response` 필드에 저장
- 이 패턴이 모듈 04 대화형 챗봇의 기초가 됩니다

### 설계

```
START → [chat] → END
State: {"user_input": str, "response": str}
```

In [ ]:
# LLM 노드 함수 정의
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

class ChatState(TypedDict):
    user_input: str
    response: str

def chat_node(state: ChatState) -> dict:
    """LLM을 호출해 응답을 상태에 저장하는 노드"""
    user_input = state["user_input"]
    ai_response = llm.invoke([HumanMessage(content=user_input)])
    return {"response": ai_response.content}

print('[OK] LLM 노드 함수 정의 완료')

In [ ]:
# LLM 그래프 조립 및 실행
chat_graph = StateGraph(ChatState)
chat_graph.add_node("chat", chat_node)
chat_graph.set_entry_point("chat")
chat_graph.add_edge("chat", END)
chat_app = chat_graph.compile()

print('[OK] LLM 그래프 컴파일 완료')
print()

# 실행
result = chat_app.invoke({"user_input": "LangGraph를 한 문장으로 설명해주세요.", "response": ""})

print('=== 실행 결과 ===')
print(f'질문: {result["user_input"]}')
print(f'응답: {result["response"]}')

---

## 섹션 5: 그래프 구조 확인

그래프는 `get_graph().print_ascii()`로 구조를 텍스트로 출력할 수 있습니다.

복잡한 그래프를 만들수록 이 시각화가 유용해집니다. 노드가 몇 개인지, 어떤 순서로 연결되어 있는지 한눈에 파악할 수 있습니다.

In [ ]:
# 그래프 시각화
print('=== 첫 번째 그래프 구조 ===')
app.get_graph().print_ascii()

print()
print('=== LLM 그래프 구조 ===')
chat_app.get_graph().print_ascii()

---

## 마무리

이 노트북에서 배운 내용을 정리합니다.

### 핵심 패턴 요약

```python
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. 상태 정의
class State(TypedDict):
    message: str

# 2. 노드 함수 작성
def my_node(state: State) -> dict:
    return {"message": "처리 완료"}

# 3. 그래프 조립
graph = StateGraph(State)
graph.add_node("my_node", my_node)
graph.set_entry_point("my_node")
graph.add_edge("my_node", END)

# 4. 컴파일 + 실행
app = graph.compile()
result = app.invoke({"message": ""})
```

### 핵심 API 레퍼런스

| API | 설명 |
|-----|------|
| `StateGraph(State)` | 그래프 생성 (상태 타입 지정) |
| `graph.add_node("이름", 함수)` | 노드 등록 |
| `graph.set_entry_point("이름")` | 진입점 지정 |
| `graph.add_edge("A", "B")` | A→B 엣지 추가 |
| `graph.add_edge("A", END)` | 종료 엣지 |
| `graph.compile()` | 실행 가능 객체 반환 |
| `app.invoke(dict)` | 그래프 실행 |
| `app.get_graph().print_ascii()` | 구조 시각화 |

---

## 다음 모듈 예고: 모듈 02 - 상태(State) 관리

모듈 01에서는 단순한 `str` 필드 하나를 가진 상태를 사용했습니다. 모듈 02에서는 상태를 더 깊이 다룹니다.

- TypedDict로 다양한 필드 타입 관리
- 여러 노드가 같은 상태를 어떻게 공유하는지
- `Annotated`와 `operator.add`로 리스트 필드 누적

```python
# 모듈 02 미리보기
import operator
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list, operator.add]  # 리스트가 덮어쓰기 대신 누적됨
    count: int
```

---

### 자기 점검 체크리스트

- [ ] `StateGraph` 4단계 패턴을 순서대로 설명할 수 있나요?
- [ ] 노드 함수가 State를 받아 dict를 반환하는 이유를 아나요?
- [ ] LLM 그래프 실행에서 `[OK]` 결과가 나왔나요?